In [1]:
# Import Libraries
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
nltk.download('vader_lexicon')
nltk.download('stopwords')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [2]:
# Define Stopwords and Clean Text Function
stop_words = set([
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'yourself','yourselves','he','him','his','himself','she','her','hers',
    'herself','it','its','itself','they','them','their','theirs','themselves',
    'what','which','who','whom','this','that','these','those','am','is','are',
    'was','were','be','been','being','have','has','had','having','do','does',
    'did','doing','a','an','the','and','but','if','or','because','as','until',
    'while','of','at','by','for','with','about','against','between','into',
    'through','during','before','after','above','below','to','from','up','down',
    'in','out','on','off','over','under','again','further','then','once','here',
    'there','when','where','why','how','all','both','each','few','more','most',
    'other','some','such','no','nor','not','only','own','same','so','than','too',
    'very','s','t','can','will','just','don','should','now','d','ll','m','o',
    're','ve','y'
])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^\w\s]", "", text)
    words = [w for w in text.split() if w not in stop_words]
    return " ".join(words)

In [3]:
# Load and Preview Education Dataset
edu_df = pd.read_csv("colleges_dataset.csv", encoding="latin1")

print("Shape:", edu_df.shape)
print("\nColumns:", edu_df.columns.tolist())
print("\nFirst 5 rows:")
edu_df.head()

Shape: (855, 4)

Columns: ['University/College', 'Ratings', 'Reviews', 'Source']

First 5 rows:


,University/College,Ratings,Reviews,Source
0,Kurnool Medical College,3.7,The faculty here is very good and well experie...,Collegedunia
1,"Ewing Christian College, Allahabad",3.2,Enjoyable college life I would give the infras...,Career360
2,Deogiri Institute of Technology and Management...,1.5,Here academics is good the focus on academics ...,Collegedunia
3,"PES University, Bangalore",1.5,1st of all they complete 2 years course in a s...,Collegedunia
4,"Tapasya College of Commerce and Management, Hy...",3.7,It good The college has good faculty and suppo...,Career360


In [4]:
#  Clean Education Reviews
edu_df['clean_review'] = edu_df['Reviews'].apply(clean_text)

print("Clean review added successfully")
print("\nSample original review:")
print(edu_df['Reviews'].iloc[0][:200])
print("\nSample clean review:")
print(edu_df['clean_review'].iloc[0][:200])

Clean review added successfully

Sample original review:
The faculty here is very good and well experienced. They too are once mbbs students so they understands the problems and difficult in understanding the subject so they teach accordingly. And every dep

Sample clean review:
faculty good well experienced mbbs students understands problems difficult understanding subject teach accordingly every department sufficient amount professors coming exams main final exam university


In [7]:
# Load Kaggle Dataset
kaggle_df = pd.read_csv("fake _reviews_dataset.csv", encoding="latin1")

print("Kaggle dataset shape:", kaggle_df.shape)
print("\nColumns:", kaggle_df.columns.tolist())
print("\nLabel distribution:")
print(kaggle_df['label'].value_counts())

Kaggle dataset shape: (40432, 4)

Columns: ['category', 'rating', 'label', 'review']

Label distribution:
label
CG    20216
OR    20216
Name: count, dtype: int64


In [8]:
#  Remove Duplicates From Kaggle
kaggle_df = kaggle_df.drop_duplicates(subset=['review'])

print("Shape after removing duplicates:", kaggle_df.shape)

Shape after removing duplicates: (40412, 4)


In [9]:
# Clean Kaggle Reviews
kaggle_df['clean_review'] = kaggle_df['review'].apply(clean_text)

print("Kaggle reviews cleaned successfully")
print("\nSample clean Kaggle review:")
print(kaggle_df['clean_review'].iloc[0][:200])

Kaggle reviews cleaned successfully

Sample clean Kaggle review:
love well made sturdy comfortable love itvery pretty


In [10]:
#  Filter Education Related Reviews From Kaggle
education_keywords = [
    "college", "university", "campus", "faculty", "professor",
    "teacher", "course", "classes", "students", "education",
    "degree", "lecture", "department", "placement",
    "internship", "hostel", "library", "lab", "exam", "semester"
]

def is_education_related(text):
    return any(word in str(text).lower() for word in education_keywords)

kaggle_filtered = kaggle_df[kaggle_df['clean_review'].apply(is_education_related)].copy()

print("Shape after education filtering:", kaggle_filtered.shape)

Shape after education filtering: (2072, 5)


In [11]:
# Map Labels OR and CG to Genuine and Fake
kaggle_filtered['label'] = kaggle_filtered['label'].map({
    "OR": "Genuine",
    "CG": "Fake"
})

print("Label distribution after mapping:")
print(kaggle_filtered['label'].value_counts())

Label distribution after mapping:
label
Genuine    1476
Fake        596
Name: count, dtype: int64


In [12]:
#  TF-IDF Vectorization
vectorizer = TfidfVectorizer(max_features=5000)

X = kaggle_filtered['clean_review']
y = kaggle_filtered['label']

X_vectorized = vectorizer.fit_transform(X)

print("Vectorized shape:", X_vectorized.shape)
print("Total features:", X_vectorized.shape[1])

Vectorized shape: (2072, 5000)
Total features: 5000


In [13]:
# Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_vectorized,
    y,
    test_size=0.2,
    random_state=42
)

print("Training size :", X_train.shape[0])
print("Testing size  :", X_test.shape[0])

Training size : 1657
Testing size  : 415


In [14]:
# Train Logistic Regression Model
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

print("Model training complete!")

Model training complete!


In [15]:
#  Evaluate Model
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy :", round(accuracy * 100, 2), "%")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy : 90.84 %

Detailed Classification Report:
              precision    recall  f1-score   support

        Fake       0.86      0.82      0.84       119
     Genuine       0.93      0.95      0.94       296

    accuracy                           0.91       415
   macro avg       0.89      0.88      0.89       415
weighted avg       0.91      0.91      0.91       415



In [16]:
#  Predict Fake Reviews on Education Dataset
edu_features = vectorizer.transform(edu_df['clean_review'])
edu_df['Fake_Prediction'] = model.predict(edu_features)

print("Prediction distribution:")
print(edu_df['Fake_Prediction'].value_counts())
print("\nFake review percentage:",
      round((edu_df['Fake_Prediction'] == 'Fake').sum() / len(edu_df) * 100, 2), "%")

Prediction distribution:
Fake_Prediction
Genuine    738
Fake       117
Name: count, dtype: int64

Fake review percentage: 13.68 %


In [17]:
#  Preview Final Output
final_df = edu_df[[
    'University/College',
    'Ratings',
    'Reviews',
    'Source',
    'clean_review',
    'Fake_Prediction'
]]

print("Final dataset shape:", final_df.shape)
print("\nGenuine reviews :", (final_df['Fake_Prediction'] == 'Genuine').sum())
print("Fake reviews    :", (final_df['Fake_Prediction'] == 'Fake').sum())
print("\nPreview:")
final_df.head(10)

Final dataset shape: (855, 6)

Genuine reviews : 738
Fake reviews    : 117

Preview:


,University/College,Ratings,Reviews,Source,clean_review,Fake_Prediction
0,Kurnool Medical College,3.7,The faculty here is very good and well experie...,Collegedunia,faculty good well experienced mbbs students un...,Genuine
1,"Ewing Christian College, Allahabad",3.2,Enjoyable college life I would give the infras...,Career360,enjoyable college life would give infrastructu...,Genuine
2,Deogiri Institute of Technology and Management...,1.5,Here academics is good the focus on academics ...,Collegedunia,academics good focus academics somewhat person...,Genuine
3,"PES University, Bangalore",1.5,1st of all they complete 2 years course in a s...,Collegedunia,st complete years course single year main focu...,Genuine
4,"Tapasya College of Commerce and Management, Hy...",3.7,It good The college has good faculty and suppo...,Career360,good college good faculty supportive teachers ...,Fake
5,"Immanuel College, Dimapur",3.7,My delights is to REVIEW my College The Colleg...,Career360,delights review college college known high tec...,Fake
6,GL Bajaj,5.0,THE APPROX FACULTY TO STUDENT RATIO IS 1:20 TO...,Collegedunia,approx faculty student ratio allow resonable a...,Genuine
7,Delhi Metropolitan Education Noida,5.0,"DME has to main annual festivals, Aloha and Vr...",Collegedunia,dme main annual festivals aloha vritika huge i...,Genuine
8,"BMS Institute of Technology and Management, Ba...",2.9,Overall good in academics and facilities are p...,Career360,overall good academics facilities provided col...,Genuine
9,"Government Women's Polytechnic, Muzaffarpur",3.2,funful and growing The college has all the nec...,Career360,funful growing college necessary facilities li...,Genuine


In [24]:
# Verify the fake prediction distribution from Milestone 2
# This confirms how many reviews were detected as Fake vs Genuine
print("Fake Prediction Distribution:")
print(edu_df['Fake_Prediction'].value_counts())

print("\nFake review percentage:",
      round((edu_df['Fake_Prediction'] == 'Fake').sum() / len(edu_df) * 100, 2), "%")

Fake Prediction Distribution:
Fake_Prediction
Genuine    738
Fake       117
Name: count, dtype: int64

Fake review percentage: 13.68 %


In [19]:
# VADER (Valence Aware Dictionary and sEntiment Reasoner) is specifically
# designed for social media and short review text sentiment analysis
# It returns a compound score between -1.0 (most negative) and 1.0 (most positive)
sia = SentimentIntensityAnalyzer()

# Test VADER on a sample sentence to verify it is working
sample = "This college has great faculty and excellent placement opportunities"
score = sia.polarity_scores(sample)
print("Sample sentence:", sample)
print("Sentiment scores:", score)
print("Compound score:", score['compound'])

Sample sentence: This college has great faculty and excellent placement opportunities
Sentiment scores: {'neg': 0.0, 'neu': 0.366, 'pos': 0.634, 'compound': 0.886}
Compound score: 0.886


In [22]:
# Apply VADER sentiment analyzer on each review
# The compound score is the overall sentiment score
# Score > 0.05  = Positive
# Score < -0.05 = Negative
# Score between -0.05 and 0.05 = Neutral
edu_df['sentiment_score'] =  edu_df['Reviews'].apply(
    lambda x: round(sia.polarity_scores(str(x))['compound'], 4)
)

print("Sentiment score added successfully")
print("\nSentiment score range:")
print("Max:",  edu_df['sentiment_score'].max())
print("Min:",  edu_df['sentiment_score'].min())
print("\nSample scores:")
print( edu_df[['University/College', 'sentiment_score']].head(10))

Sentiment score added successfully

Sentiment score range:
Max: 0.9987
Min: -0.9932

Sample scores:
                                  University/College  sentiment_score
0                            Kurnool Medical College           0.0917
1                 Ewing Christian College, Allahabad           0.9719
2  Deogiri Institute of Technology and Management...           0.4404
3                          PES University, Bangalore          -0.3875
4  Tapasya College of Commerce and Management, Hy...           0.9797
5                          Immanuel College, Dimapur           0.9937
6                                           GL Bajaj           0.3415
7                 Delhi Metropolitan Education Noida           0.8689
8  BMS Institute of Technology and Management, Ba...           0.7835
9        Government Women's Polytechnic, Muzaffarpur           0.9323


In [25]:
# Assign Positive, Negative or Neutral label
# based on the compound sentiment score from VADER
def sentiment_label(score):
    if score > 0.05:
        return "Positive"
    elif score < -0.05:
        return "Negative"
    else:
        return "Neutral"

edu_df['Sentiment'] = edu_df['sentiment_score'].apply(sentiment_label)

print("Sentiment labels assigned successfully")
print("\nSentiment distribution:")
print(edu_df['Sentiment'].value_counts())
print("\nSentiment percentage:")
print(round(edu_df['Sentiment'].value_counts() / len(edu_df) * 100, 2))

Sentiment labels assigned successfully

Sentiment distribution:
Sentiment
Positive    725
Negative    109
Neutral      21
Name: count, dtype: int64

Sentiment percentage:
Sentiment
Positive    84.80
Negative    12.75
Neutral      2.46
Name: count, dtype: float64


In [26]:
# Cross analysis of Sentiment vs Fake Prediction
# This shows whether fake reviews tend to be more positive or negative
# compared to genuine reviews — a key insight for the project
print("Sentiment vs Fake Prediction cross analysis:")
print(pd.crosstab(edu_df['Fake_Prediction'], edu_df['Sentiment']))

print("\nPercentage breakdown:")
print(pd.crosstab(edu_df['Fake_Prediction'], edu_df['Sentiment'], normalize='index').round(3) * 100)

Sentiment vs Fake Prediction cross analysis:
Sentiment        Negative  Neutral  Positive
Fake_Prediction                             
Fake                    7        1       109
Genuine               102       20       616

Percentage breakdown:
Sentiment        Negative  Neutral  Positive
Fake_Prediction                             
Fake                  6.0      0.9      93.2
Genuine              13.8      2.7      83.5


In [27]:
# Check sentiment distribution across Career360 and Collegedunia
# This shows if one platform has more positive or negative reviews
print("Sentiment distribution by Source:")
print(pd.crosstab(edu_df['Source'], edu_df['Sentiment']))

print("\nPercentage breakdown by source:")
print(pd.crosstab(edu_df['Source'], edu_df['Sentiment'], normalize='index').round(3) * 100)

Sentiment distribution by Source:
Sentiment     Negative  Neutral  Positive
Source                                   
Career360           36        0       464
Collegedunia        73       21       261

Percentage breakdown by source:
Sentiment     Negative  Neutral  Positive
Source                                   
Career360          7.2      0.0      92.8
Collegedunia      20.6      5.9      73.5


In [28]:
# Calculate average sentiment score for each platform
# Higher score means more positive reviews on that platform
print("Average sentiment score by Source:")
print(edu_df.groupby('Source')['sentiment_score'].mean().round(4))

print("\nAverage sentiment score by Fake Prediction:")
print(edu_df.groupby('Fake_Prediction')['sentiment_score'].mean().round(4))

Average sentiment score by Source:
Source
Career360       0.8234
Collegedunia    0.4345
Name: sentiment_score, dtype: float64

Average sentiment score by Fake Prediction:
Fake_Prediction
Fake       0.8532
Genuine    0.6316
Name: sentiment_score, dtype: float64


In [29]:
# Display the top 5 most positive reviews based on sentiment score
# These are likely genuine enthusiastic student reviews
print("Top 5 Most Positive Reviews:")
top_positive = edu_df.nlargest(5, 'sentiment_score')[
    ['University/College', 'Reviews', 'sentiment_score', 'Sentiment', 'Fake_Prediction']
]
for i, row in top_positive.iterrows():
    print(f"\nCollege : {row['University/College']}")
    print(f"Score   : {row['sentiment_score']}")
    print(f"Label   : {row['Fake_Prediction']}")
    print(f"Review  : {str(row['Reviews'])[:150]}")
    print("-" * 60)

Top 5 Most Positive Reviews:

College : SAM Global University, Bhopal
Score   : 0.9987
Label   : Genuine
Review  : Good experience The college infrastructure is decent and properly maintained. Classrooms are well arranged with modern teaching aids that help in bett
------------------------------------------------------------

College : Asian Academy of Film and Television, Noida
Score   : 0.9987
Label   : Genuine
Review  : Its a decent college The college infrastructure is adequate and fulfills the essential needs of students. Classrooms are functional with sufficient se
------------------------------------------------------------

College : BJB Autonomous College, Bhubaneswar
Score   : 0.998
Label   : Genuine
Review  : Out of 10 I rate my college 6. Good infrastructure but need to improve a little. Need to improve the food hygiene level a little more. Overall it's fi
------------------------------------------------------------

College : ITM School of Pharmacy, Vadodara
Score   : 0.99

In [30]:
# Display the top 5 most negative reviews based on sentiment score
# These reviews likely contain strong complaints or dissatisfaction
print("Top 5 Most Negative Reviews:")
top_negative = edu_df.nsmallest(5, 'sentiment_score')[
    ['University/College', 'Reviews', 'sentiment_score', 'Sentiment', 'Fake_Prediction']
]
for i, row in top_negative.iterrows():
    print(f"\nCollege : {row['University/College']}")
    print(f"Score   : {row['sentiment_score']}")
    print(f"Label   : {row['Fake_Prediction']}")
    print(f"Review  : {str(row['Reviews'])[:150]}")
    print("-" * 60)

Top 5 Most Negative Reviews:

College : RNS Institute of Technology, Bangalore
Score   : -0.9932
Label   : Genuine
Review  : This review is useless since it is not written by choice. their constructing all the time. computers are fine. classrooms are fine This review is usel
------------------------------------------------------------

College : Unique College, Bhopal
Score   : -0.9897
Label   : Genuine
Review  : okay Wi-Fi is not available in this college. Labs are very small, and the classroom infrastructure is very bad. Libraries are very small. Hostel facil
------------------------------------------------------------

College : Mumbai University
Score   : -0.9838
Label   : Genuine
Review  : I opted for this college because I saw that an integrated course of Management with MBA is available. I don't like anything about this college. It's t
------------------------------------------------------------

College : Disha College, Raipur
Score   : -0.9836
Label   : Genuine
Review  : Bad e

In [31]:
# Display the top 5 most negative reviews based on sentiment score
# These reviews likely contain strong complaints or dissatisfaction
print("Top 5 Most Negative Reviews:")
top_negative = edu_df.nsmallest(5, 'sentiment_score')[
    ['University/College', 'Reviews', 'sentiment_score', 'Sentiment', 'Fake_Prediction']
]
for i, row in top_negative.iterrows():
    print(f"\nCollege : {row['University/College']}")
    print(f"Score   : {row['sentiment_score']}")
    print(f"Label   : {row['Fake_Prediction']}")
    print(f"Review  : {str(row['Reviews'])[:150]}")
    print("-" * 60)

Top 5 Most Negative Reviews:

College : RNS Institute of Technology, Bangalore
Score   : -0.9932
Label   : Genuine
Review  : This review is useless since it is not written by choice. their constructing all the time. computers are fine. classrooms are fine This review is usel
------------------------------------------------------------

College : Unique College, Bhopal
Score   : -0.9897
Label   : Genuine
Review  : okay Wi-Fi is not available in this college. Labs are very small, and the classroom infrastructure is very bad. Libraries are very small. Hostel facil
------------------------------------------------------------

College : Mumbai University
Score   : -0.9838
Label   : Genuine
Review  : I opted for this college because I saw that an integrated course of Management with MBA is available. I don't like anything about this college. It's t
------------------------------------------------------------

College : Disha College, Raipur
Score   : -0.9836
Label   : Genuine
Review  : Bad e

In [32]:
# Compare average college ratings across sentiment categories
# This validates whether sentiment score aligns with numeric ratings
print("Average Rating by Sentiment:")
print(edu_df.groupby('Sentiment')['Ratings'].mean().round(2))

print("\nAverage Rating by Fake Prediction:")
print(edu_df.groupby('Fake_Prediction')['Ratings'].mean().round(2))

Average Rating by Sentiment:
Sentiment
Negative    2.24
Neutral     2.78
Positive    3.63
Name: Ratings, dtype: float64

Average Rating by Fake Prediction:
Fake_Prediction
Fake       3.57
Genuine    3.41
Name: Ratings, dtype: float64


In [33]:
# Extract most common words from fake reviews vs genuine reviews
# This helps identify patterns that distinguish fake from genuine reviews
from collections import Counter

# Get all words from fake reviews and genuine reviews
fake_words    = ' '.join(edu_df[edu_df['Fake_Prediction'] == 'Fake']['clean_review']).split()
genuine_words = ' '.join(edu_df[edu_df['Fake_Prediction'] == 'Genuine']['clean_review']).split()

# Count top 15 words in each category
fake_common    = Counter(fake_words).most_common(15)
genuine_common = Counter(genuine_words).most_common(15)

print("Top 15 Most Common Words in FAKE Reviews:")
for word, count in fake_common:
    print(f"  {word}: {count}")

print("\nTop 15 Most Common Words in GENUINE Reviews:")
for word, count in genuine_common:
    print(f"  {word}: {count}")

Top 15 Most Common Words in FAKE Reviews:
  college: 434
  good: 401
  students: 370
  also: 153
  placement: 130
  infrastructure: 107
  campus: 106
  well: 93
  facilities: 81
  placements: 75
  faculty: 71
  supportive: 71
  companies: 66
  opportunities: 65
  teachers: 63

Top 15 Most Common Words in GENUINE Reviews:
  college: 1336
  good: 996
  students: 916
  placement: 603
  campus: 482
  also: 434
  infrastructure: 387
  like: 371
  well: 327
  faculty: 322
  facilities: 306
  placements: 299
  student: 281
  get: 265
  many: 262


In [34]:
# Preview the final dataset with all NLP columns added
# This dataset now has 8 columns ready for Power BI import
final_nlp = edu_df[[
    'University/College',
    'Ratings',
    'Reviews',
    'Source',
    'clean_review',
    'Fake_Prediction',
    'sentiment_score',
    'Sentiment'
]]

print("Final NLP Dataset Shape:", final_nlp.shape)
print("\nColumns:", final_nlp.columns.tolist())
print("\nSentiment distribution:")
print(final_nlp['Sentiment'].value_counts())
print("\nFake Prediction distribution:")
print(final_nlp['Fake_Prediction'].value_counts())
print("\nPreview:")
final_nlp.head(10)

Final NLP Dataset Shape: (855, 8)

Columns: ['University/College', 'Ratings', 'Reviews', 'Source', 'clean_review', 'Fake_Prediction', 'sentiment_score', 'Sentiment']

Sentiment distribution:
Sentiment
Positive    725
Negative    109
Neutral      21
Name: count, dtype: int64

Fake Prediction distribution:
Fake_Prediction
Genuine    738
Fake       117
Name: count, dtype: int64

Preview:


,University/College,Ratings,Reviews,Source,clean_review,Fake_Prediction,sentiment_score,Sentiment
0,Kurnool Medical College,3.7,The faculty here is very good and well experie...,Collegedunia,faculty good well experienced mbbs students un...,Genuine,0.0917,Positive
1,"Ewing Christian College, Allahabad",3.2,Enjoyable college life I would give the infras...,Career360,enjoyable college life would give infrastructu...,Genuine,0.9719,Positive
2,Deogiri Institute of Technology and Management...,1.5,Here academics is good the focus on academics ...,Collegedunia,academics good focus academics somewhat person...,Genuine,0.4404,Positive
3,"PES University, Bangalore",1.5,1st of all they complete 2 years course in a s...,Collegedunia,st complete years course single year main focu...,Genuine,-0.3875,Negative
4,"Tapasya College of Commerce and Management, Hy...",3.7,It good The college has good faculty and suppo...,Career360,good college good faculty supportive teachers ...,Fake,0.9797,Positive
5,"Immanuel College, Dimapur",3.7,My delights is to REVIEW my College The Colleg...,Career360,delights review college college known high tec...,Fake,0.9937,Positive
6,GL Bajaj,5.0,THE APPROX FACULTY TO STUDENT RATIO IS 1:20 TO...,Collegedunia,approx faculty student ratio allow resonable a...,Genuine,0.3415,Positive
7,Delhi Metropolitan Education Noida,5.0,"DME has to main annual festivals, Aloha and Vr...",Collegedunia,dme main annual festivals aloha vritika huge i...,Genuine,0.8689,Positive
8,"BMS Institute of Technology and Management, Ba...",2.9,Overall good in academics and facilities are p...,Career360,overall good academics facilities provided col...,Genuine,0.7835,Positive
9,"Government Women's Polytechnic, Muzaffarpur",3.2,funful and growing The college has all the nec...,Career360,funful growing college necessary facilities li...,Genuine,0.9323,Positive


In [35]:
# Save the final complete dataset with all columns
# This file will be imported into Power BI for dashboard creation
final_nlp.to_csv("education_reviews_final.csv", index=False)

print("File saved as education_reviews_final.csv")
print("\nFinal Summary:")
print(f"Total reviews      : {len(final_nlp)}")
print(f"Genuine reviews    : {(final_nlp['Fake_Prediction']=='Genuine').sum()}")
print(f"Fake reviews       : {(final_nlp['Fake_Prediction']=='Fake').sum()}")
print(f"Positive sentiment : {(final_nlp['Sentiment']=='Positive').sum()}")
print(f"Negative sentiment : {(final_nlp['Sentiment']=='Negative').sum()}")
print(f"Neutral sentiment  : {(final_nlp['Sentiment']=='Neutral').sum()}")
print(f"\nFile ready for Power BI import!")

File saved as education_reviews_final.csv

Final Summary:
Total reviews      : 855
Genuine reviews    : 738
Fake reviews       : 117
Positive sentiment : 725
Negative sentiment : 109
Neutral sentiment  : 21

File ready for Power BI import!
